# ZI-SOHO Phase A — CIFAR-100 train-only gate
This notebook evaluates the zero-inflated fixed-WTA hypothesis without opening `test.pt`. Run all cells in order. The WTA code cache is experiment infrastructure and is never learner state.

In [ ]:
# === Edit paths only. Do not edit model/search parameters. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/zi-soho'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_FEATURE_CACHE = f'{DRIVE_ROOT}/tsoho_cifar100_cache'
FEATURE_CACHE_DIR = '/content/tsoho_cifar100_cache'
# About 0.9 GB. Keeping this on Drive makes interrupted reruns cheaper.
WTA_CODE_CACHE_DIR = f'{DRIVE_ROOT}/zi_soho_wta_h10000_seed1993'
OUTPUT_DIR = f'{DRIVE_ROOT}/zi_soho_phasea_train_only_outputs'
CHECKPOINT_SOURCE = 'huggingface'  # or 'google_drive'
DRIVE_CHECKPOINT_PATH = f'{DRIVE_ROOT}/model.safetensors'
BATCH_SIZE = 128
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'

In [ ]:
# Runtime setup.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
shutil.rmtree(WORK_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_GIT_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
print('repo commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Restore the shared frozen-feature cache from Drive when available.
drive_cache = Path(DRIVE_FEATURE_CACHE)
local_cache = Path(FEATURE_CACHE_DIR)
core = ('metadata.json', 'train.pt')
has_drive_cache = all((drive_cache/name).is_file() for name in core) and ((drive_cache/'test.pt').is_file() or (drive_cache/'test.locked.pt').is_file())
if has_drive_cache:
    shutil.rmtree(local_cache, ignore_errors=True)
    started = time.time()
    shutil.copytree(drive_cache, local_cache)
    print(f'Feature cache restored: {time.time()-started:.1f}s')
else:
    print('No complete Drive feature cache. Cell 5 will extract it once.')

In [ ]:
# Extract frozen ViT features only if restore was unavailable. Live output is one line per task.
local_cache = Path(FEATURE_CACHE_DIR)
has_local_cache = all((local_cache/name).is_file() for name in ('metadata.json','train.pt')) and ((local_cache/'test.pt').is_file() or (local_cache/'test.locked.pt').is_file())
if not has_local_cache:
    shutil.rmtree(local_cache, ignore_errors=True)
    if CHECKPOINT_SOURCE == 'google_drive':
        CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
    elif CHECKPOINT_SOURCE == 'huggingface':
        from huggingface_hub import hf_hub_download
        CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
    else:
        raise ValueError("CHECKPOINT_SOURCE must be 'huggingface' or 'google_drive'")
    import kagglehub
    downloaded = Path(kagglehub.dataset_download('zaphat206/cifar-100'))
    candidates = [downloaded, *downloaded.rglob('cifar-100')]
    cifar = next(path for path in candidates if (path/'train').is_file() and (path/'test').is_file() and (path/'meta').is_file())
    command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--root', str(cifar), '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', FEATURE_CACHE_DIR, '--output-dir', '/content/zi_soho_extract', '--dataset', 'CIFAR-100', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', '1993', '--num-classes', '100', '--num-tasks', '10', '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
    print('FEATURE EXTRACT start — wait for 20 train/test task lines.', flush=True)
    subprocess.run(command, check=True)
    shutil.rmtree(drive_cache, ignore_errors=True)
    shutil.copytree(local_cache, drive_cache)
    print('Feature cache saved to Drive:', drive_cache)
else:
    print('Using restored feature cache; extraction skipped.')
metadata = json.loads((local_cache/'metadata.json').read_text())
assert metadata['feature_dim'] == 768 and metadata['finite'] is True
assert metadata['checkpoint_sha256'] == CHECKPOINT_SHA256
print('FEATURE CACHE PASS:', metadata['train_shape'], metadata['dtype'])

In [ ]:
# Focused correctness gate. Sparse-CSC beta warnings are expected and non-fatal.
command = [sys.executable, '-m', 'pytest', '-q', 'tests/test_zi_soho_math.py', 'tests/test_zi_soho_learner.py', 'tests/test_zi_soho_pilot.py']
print('TEST start:', ' '.join(command), flush=True)
completed = subprocess.run(command)
assert completed.returncode == 0, 'ZI-SOHO tests failed. Stop and send the full traceback.'
print('ZI-SOHO correctness gate: PASS')

In [ ]:
# Physically hide held-out test, then run/resume 10 train-only candidates.
test_path = local_cache/'test.pt'
locked_test_path = local_cache/'test.locked.pt'
if test_path.is_file():
    test_path.replace(locked_test_path)
assert locked_test_path.is_file() and not test_path.exists(), 'test.pt must be hidden before selection'
command = [sys.executable, '-u', 'tools/zi_soho_pilot.py', '--config', 'configs/zi_soho_cifar100_train_only.json', '--feature-cache-dir', FEATURE_CACHE_DIR, '--code-cache-dir', WTA_CODE_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-test-hidden', '--resume']
print('ZI-SOHO TRAIN-ONLY start.', flush=True)
print('First run: WTA CACHE lines show percent/ETA; then START/UPDATE/DONE show candidate progress.', flush=True)
print('A FLY UPDATE can stay visible during its 10000x10000 solve; that alone is not a freeze.', flush=True)
started = time.time()
completed = subprocess.run(command)
assert completed.returncode == 0, 'Pilot failed. Stop and send the full traceback; do not edit parameters.'
print(f'ZI-SOHO train-only pilot COMPLETE in {(time.time()-started)/60:.1f} minutes')

In [ ]:
# Compact report. This cell does not restore or open test.pt.
import pandas as pd
payload = json.loads(Path(OUTPUT_DIR, 'selection.json').read_text())
rows = list(payload['best_by_method'].values())
table = pd.DataFrame(rows).sort_values('validation_average_accuracy', ascending=False)
columns = ['method','variance_kappa','ridge_lambda','ridge_lower','ridge_upper','validation_average_accuracy','persistent_state_bytes','candidate_seconds']
display(table[[column for column in columns if column in table.columns]])
print(json.dumps(payload['gate'], indent=2))
print('DECISION:', payload['gate']['decision'])
print('Held-out test authorized:', payload['held_out_test_authorized'])
assert locked_test_path.is_file() and not test_path.exists()

In [ ]:
# Download only the train-only evidence; the feature/WTA caches are excluded.
artifact = shutil.make_archive('/content/zi_soho_phasea_train_only', 'zip', OUTPUT_DIR)
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()
print('artifact:', artifact)
print('SHA-256:', sha256(artifact))
from google.colab import files
files.download(artifact)